In [ ]:
import cv2
import matplotlib.pyplot as plt

image_path = "images/CFD_001.jpg"
mask_path = "masks/CFD_001.jpg"

image = cv2.imread(image_path)
mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

# OpenCV uses BGR, matplotlib uses RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)


plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Crack Mask")
plt.axis("off")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------
# 1. Load image
# --------------------------------

image = cv2.imread("images/CFD_001.jpg")

# --------------------------------
# 2. Load ground-truth mask
# --------------------------------

ground_truth = cv2.imread(
    "masks/CFD_001.jpg",
    cv2.IMREAD_GRAYSCALE
)

_, ground_truth = cv2.threshold(
    ground_truth,
    127,
    255,
    cv2.THRESH_BINARY
)

# --------------------------------
# 3. Create prediction
# --------------------------------

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

_, prediction = cv2.threshold(
    gray,
    100,
    255,
    cv2.THRESH_BINARY_INV
)

# --------------------------------
# 4. Calculate TP, FP, FN
# --------------------------------

TP = np.logical_and(
    ground_truth == 255,
    prediction == 255
).sum()

FP = np.logical_and(
    ground_truth == 0,
    prediction == 255
).sum()

FN = np.logical_and(
    ground_truth == 255,
    prediction == 0
).sum()

# --------------------------------
# 5. Calculate IoU
# --------------------------------

IoU = TP / (TP + FP + FN)

print("TP:", TP)
print("FP:", FP)
print("FN:", FN)
print("IoU:", IoU)
print("IoU (%):", IoU * 100)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# ========================================
# SETTINGS
# ========================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)

NUMBER_OF_IMAGES = 10       # <-- change this number
THRESHOLD = 100             # threshold value

valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ========================================
# GET IMAGE FILES
# ========================================

image_paths = sorted([
    path for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])


# ========================================
# PROCESS SELECTED NUMBER OF IMAGES
# ========================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    # ------------------------------------
    # Find corresponding mask
    # ------------------------------------

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print(f"Mask not found: {image_path.name}")
        continue

    # ------------------------------------
    # Read image
    # ------------------------------------

    image = cv2.imread(str(image_path))

    # ------------------------------------
    # Read ground-truth mask
    # ------------------------------------

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print(f"Could not read: {image_path.name}")
        continue

    # ------------------------------------
    # Convert image to grayscale
    # ------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # ------------------------------------
    # Threshold original image
    # ------------------------------------

    _, binary = cv2.threshold(
        gray,
        THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )

    # ------------------------------------
    # Make mask binary
    # ------------------------------------

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # ====================================
    # CALCULATE IoU
    # ====================================

    TP = ((mask_binary == 255) & (binary == 255)).sum()

    FP = ((mask_binary == 0) & (binary == 255)).sum()

    FN = ((mask_binary == 255) & (binary == 0)).sum()

    denominator = TP + FP + FN

    if denominator > 0:
        IoU = TP / denominator
    else:
        IoU = 0

    # ------------------------------------
    # Print result
    # ------------------------------------

    print(
        f"{image_path.name} | "
        f"TP={TP} | "
        f"FP={FP} | "
        f"FN={FN} | "
        f"IoU={IoU:.4f} "
        f"({IoU * 100:.2f}%)"
    )

    # ====================================
    # DISPLAY
    # ====================================

    plt.figure(figsize=(15, 5))

    # Original image
    plt.subplot(1, 3, 1)

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.title(f"Original\n{image_path.name}")
    plt.axis("off")

    # Binary prediction
    plt.subplot(1, 3, 2)

    plt.imshow(
        binary,
        cmap="gray"
    )

    plt.title("Binary Prediction")
    plt.axis("off")

    # Ground-truth mask
    plt.subplot(1, 3, 3)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title(f"Ground Truth Mask\nIoU = {IoU:.2%}")
    plt.axis("off")

    plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)

NUMBER_OF_IMAGES = 10

# Simple threshold
THRESHOLD = 99

# Median filter AFTER threshold
MEDIAN_KERNEL = 3

# Erosion
EROSION_KERNEL = 3
EROSION_ITERATIONS = 1


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # ========================================================
    # FIND MASK
    # ========================================================

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # ========================================================
    # READ IMAGE AND MASK
    # ========================================================

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # ========================================================
    # 1. GRAYSCALE
    # ========================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # ========================================================
    # 2. SIMPLE THRESHOLD
    # ========================================================

    _, binary = cv2.threshold(
        gray,
        THRESHOLD,
        255,
        cv2.THRESH_BINARY_INV
    )

    # ========================================================
    # 3. MEDIAN FILTER
    # ========================================================

    median_binary = cv2.medianBlur(
        binary,
        MEDIAN_KERNEL
    )

    # ========================================================
    # 4. EROSION
    # ========================================================

    erosion_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (EROSION_KERNEL, EROSION_KERNEL)
    )

    eroded = cv2.erode(
        median_binary,
        erosion_kernel,
        iterations=EROSION_ITERATIONS
    )

    # ========================================================
    # 5. GROUND TRUTH MASK
    # ========================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # ========================================================
    # 6. CALCULATE IoU
    # ========================================================

    TP = np.logical_and(
        mask_binary == 255,
        eroded == 255
    ).sum()

    FP = np.logical_and(
        mask_binary == 0,
        eroded == 255
    ).sum()

    FN = np.logical_and(
        mask_binary == 255,
        eroded == 0
    ).sum()

    denominator = TP + FP + FN

    IoU = (
        TP / denominator
        if denominator > 0
        else 0
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("TP:", TP)
    print("FP:", FP)
    print("FN:", FN)

    print(
        f"IoU: {IoU:.4f} "
        f"({IoU * 100:.2f}%)"
    )

    # ========================================================
    # DISPLAY ALL STAGES
    # ========================================================

    plt.figure(figsize=(20, 5))

    # --------------------------------------------------------
    # 1. ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 6, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"1. Original\n{image_path.name}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 2. GRAYSCALE
    # --------------------------------------------------------

    plt.subplot(1, 6, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("2. Grayscale")

    plt.axis("off")

    # --------------------------------------------------------
    # 3. SIMPLE THRESHOLD
    # --------------------------------------------------------

    plt.subplot(1, 6, 3)

    plt.imshow(
        binary,
        cmap="gray"
    )

    plt.title(
        f"3. Simple Threshold\n"
        f"Threshold={THRESHOLD}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 4. MEDIAN FILTER
    # --------------------------------------------------------

    plt.subplot(1, 6, 4)

    plt.imshow(
        median_binary,
        cmap="gray"
    )

    plt.title(
        f"4. Median Filter\n"
        f"Kernel={MEDIAN_KERNEL}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 5. EROSION
    # --------------------------------------------------------

    plt.subplot(1, 6, 5)

    plt.imshow(
        eroded,
        cmap="gray"
    )

    plt.title(
        f"5. Erosion\n"
        f"Kernel={EROSION_KERNEL}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # 6. GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 6, 6)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title(
        f"6. Ground Truth\n"
        f"IoU={IoU:.2%}"
    )

    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)

NUMBER_OF_IMAGES = 10

# Test every threshold
THRESHOLDS = range(0, 256)


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# IoU FUNCTION
# ============================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# ============================================================
# STORE RESULTS
# ============================================================

all_iou_results = []

best_thresholds = []
best_ious = []


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # --------------------------------------------------------
    # FIND MASK
    # --------------------------------------------------------

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # --------------------------------------------------------
    # READ IMAGE
    # --------------------------------------------------------

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # --------------------------------------------------------
    # BINARY GROUND TRUTH
    # --------------------------------------------------------

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # --------------------------------------------------------
    # TEST ALL THRESHOLDS
    # --------------------------------------------------------

    iou_values = []

    for threshold in THRESHOLDS:

        _, prediction = cv2.threshold(
            gray,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )

        iou = calculate_iou(
            prediction,
            mask_binary
        )

        iou_values.append(iou)

    # --------------------------------------------------------
    # FIND BEST THRESHOLD
    # --------------------------------------------------------

    best_index = np.argmax(iou_values)

    best_threshold = list(THRESHOLDS)[best_index]
    best_iou = iou_values[best_index]

    # Save results
    all_iou_results.append(iou_values)

    best_thresholds.append(best_threshold)
    best_ious.append(best_iou)

    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print("Best threshold:", best_threshold)

    print(
        f"Best IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )

    # ========================================================
    # CREATE BEST PREDICTION
    # ========================================================

    _, best_prediction = cv2.threshold(
        gray,
        best_threshold,
        255,
        cv2.THRESH_BINARY_INV
    )

    # ========================================================
    # DISPLAY
    # ========================================================

    plt.figure(figsize=(18, 5))

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 4, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"Original\n{image_path.name}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    plt.subplot(1, 4, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")

    # --------------------------------------------------------
    # BEST PREDICTION
    # --------------------------------------------------------

    plt.subplot(1, 4, 3)

    plt.imshow(
        best_prediction,
        cmap="gray"
    )

    plt.title(
        f"Best Threshold = {best_threshold}\n"
        f"IoU = {best_iou:.2%}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 4, 4)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # IoU GRAPH FOR THIS IMAGE
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.plot(
        list(THRESHOLDS),
        iou_values
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.scatter(
        best_threshold,
        best_iou
    )

    plt.xlabel("Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs Threshold\n"
        f"{image_path.name}"
    )

    plt.grid()

    plt.show()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

for i, image_path in enumerate(
    image_paths[:len(best_thresholds)]
):

    print(
        f"{image_path.name:25s} "
        f"Threshold={best_thresholds[i]:3d} "
        f"IoU={best_ious[i]:.2%}"
    )


# ============================================================
# AVERAGE IoU FOR EACH THRESHOLD
# ============================================================

if len(all_iou_results) > 0:

    all_iou_results = np.array(
        all_iou_results
    )

    average_iou = np.mean(
        all_iou_results,
        axis=0
    )

    best_average_index = np.argmax(
        average_iou
    )

    best_average_threshold = best_average_index
    best_average_iou = average_iou[
        best_average_index
    ]

    print("\n" + "=" * 70)
    print("DATASET RESULT")
    print("=" * 70)

    print(
        "Best average threshold:",
        best_average_threshold
    )

    print(
        f"Best average IoU: "
        f"{best_average_iou:.4f} "
        f"({best_average_iou * 100:.2f}%)"
    )

    # ========================================================
    # DATASET IoU GRAPH
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.plot(
        list(THRESHOLDS),
        average_iou
    )

    plt.axvline(
        best_average_threshold,
        linestyle="--"
    )

    plt.scatter(
        best_average_threshold,
        best_average_iou
    )

    plt.xlabel("Threshold")
    plt.ylabel("Average IoU")

    plt.title(
        "Average IoU vs Threshold"
    )

    plt.grid()

    plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)

NUMBER_OF_IMAGES = 100

# Test thresholds
THRESHOLDS = range(0, 256)

# Morphological opening
OPENING_KERNEL_SIZE = 3
OPENING_ITERATIONS = 1


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# IoU FUNCTION
# ============================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# ============================================================
# MORPHOLOGICAL OPENING FUNCTION
# ============================================================

def apply_opening(binary_image):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            OPENING_KERNEL_SIZE,
            OPENING_KERNEL_SIZE
        )
    )

    opened = cv2.morphologyEx(
        binary_image,
        cv2.MORPH_OPEN,
        kernel,
        iterations=OPENING_ITERATIONS
    )

    return opened


# ============================================================
# STORE RESULTS
# ============================================================

all_iou_results = []

best_thresholds = []
best_ious = []


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # --------------------------------------------------------
    # FIND MASK
    # --------------------------------------------------------

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # --------------------------------------------------------
    # READ IMAGE
    # --------------------------------------------------------

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # --------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # --------------------------------------------------------
    # TEST ALL THRESHOLDS
    # --------------------------------------------------------

    iou_values = []

    for threshold in THRESHOLDS:

        # ====================================================
        # 1. THRESHOLD
        # ====================================================

        _, binary = cv2.threshold(
            gray,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )

        # ====================================================
        # 2. MORPHOLOGICAL OPENING
        # ====================================================

        opened = apply_opening(binary)

        # ====================================================
        # 3. IoU
        # ====================================================

        iou = calculate_iou(
            opened,
            mask_binary
        )

        iou_values.append(iou)

    # --------------------------------------------------------
    # FIND BEST THRESHOLD
    # --------------------------------------------------------

    best_index = np.argmax(iou_values)

    best_threshold = list(THRESHOLDS)[best_index]

    best_iou = iou_values[best_index]

    # Save results
    all_iou_results.append(iou_values)

    best_thresholds.append(best_threshold)
    best_ious.append(best_iou)

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(
        f"Best threshold: {best_threshold}"
    )

    print(
        f"Best IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )

    # ========================================================
    # CREATE BEST PREDICTION
    # ========================================================

    _, best_binary = cv2.threshold(
        gray,
        best_threshold,
        255,
        cv2.THRESH_BINARY_INV
    )

    best_opened = apply_opening(
        best_binary
    )

    # ========================================================
    # DISPLAY
    # ========================================================

    plt.figure(figsize=(20, 5))

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 5, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"Original\n{image_path.name}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    plt.subplot(1, 5, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")

    # --------------------------------------------------------
    # THRESHOLD
    # --------------------------------------------------------

    plt.subplot(1, 5, 3)

    plt.imshow(
        best_binary,
        cmap="gray"
    )

    plt.title(
        f"Threshold\n"
        f"T = {best_threshold}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # OPENING
    # --------------------------------------------------------

    plt.subplot(1, 5, 4)

    plt.imshow(
        best_opened,
        cmap="gray"
    )

    plt.title(
        f"Opening\n"
        f"Kernel = {OPENING_KERNEL_SIZE}\n"
        f"IoU = {best_iou:.2%}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 5, 5)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # IoU VS THRESHOLD
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.plot(
        list(THRESHOLDS),
        iou_values
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.scatter(
        best_threshold,
        best_iou
    )

    plt.xlabel("Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs Threshold\n"
        f"{image_path.name} + Opening"
    )

    plt.grid()

    plt.show()


# ============================================================
# DATASET SUMMARY
# ============================================================

if len(all_iou_results) > 0:

    all_iou_results = np.array(
        all_iou_results
    )

    average_iou = np.mean(
        all_iou_results,
        axis=0
    )

    best_average_index = np.argmax(
        average_iou
    )

    best_average_threshold = (
        best_average_index
    )

    best_average_iou = average_iou[
        best_average_index
    ]

    print("\n" + "=" * 70)
    print("DATASET RESULT")
    print("=" * 70)

    print(
        "Best average threshold:",
        best_average_threshold
    )

    print(
        f"Best average IoU: "
        f"{best_average_iou:.4f} "
        f"({best_average_iou * 100:.2f}%)"
    )

    # --------------------------------------------------------
    # DATASET GRAPH
    # --------------------------------------------------------

    plt.figure(figsize=(10, 5))

    plt.plot(
        list(THRESHOLDS),
        average_iou
    )

    plt.axvline(
        best_average_threshold,
        linestyle="--"
    )

    plt.scatter(
        best_average_threshold,
        best_average_iou
    )

    plt.xlabel("Threshold")
    plt.ylabel("Average IoU")

    plt.title(
        "Average IoU vs Threshold + Morphological Opening"
    )

    plt.grid()

    plt.show()


# ============================================================
# AVERAGE IoU OF ALL IMAGES
# ============================================================

if len(best_ious) > 0:

    average_iou = np.mean(best_ious)

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    print(
        f"Average IoU: {average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )

else:
    print("No IoU results available.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)

NUMBER_OF_IMAGES = 100

# Test thresholds
THRESHOLDS = range(0, 256)

# Morphological Closing
CLOSING_KERNEL_SIZE = 3
CLOSING_ITERATIONS = 1

# Median filter
MEDIAN_KERNEL_SIZE = 3


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# IoU FUNCTION
# ============================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# ============================================================
# MORPHOLOGICAL CLOSING
# ============================================================

def apply_closing(binary_image):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            CLOSING_KERNEL_SIZE,
            CLOSING_KERNEL_SIZE
        )
    )

    closed = cv2.morphologyEx(
        binary_image,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=CLOSING_ITERATIONS
    )

    return closed


# ============================================================
# STORE RESULTS
# ============================================================

all_iou_results = []

best_thresholds = []
best_ious = []


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # --------------------------------------------------------
    # FIND MASK
    # --------------------------------------------------------

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # --------------------------------------------------------
    # READ IMAGE
    # --------------------------------------------------------

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # ========================================================
    # 1. GRAYSCALE
    # ========================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # ========================================================
    # GROUND TRUTH
    # ========================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # ========================================================
    # TEST ALL THRESHOLDS
    # ========================================================

    iou_values = []

    for threshold in THRESHOLDS:

        # ====================================================
        # 2. THRESHOLDING
        # ====================================================

        _, binary = cv2.threshold(
            gray,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )

        # ====================================================
        # 3. CLOSING
        # ====================================================

        closed = apply_closing(binary)

        # ====================================================
        # 4. MEDIAN FILTER
        # ====================================================

        median = cv2.medianBlur(
            closed,
            MEDIAN_KERNEL_SIZE
        )

        # ====================================================
        # 5. IoU
        # ====================================================

        iou = calculate_iou(
            median,
            mask_binary
        )

        iou_values.append(iou)

    # ========================================================
    # FIND BEST THRESHOLD
    # ========================================================

    best_index = np.argmax(iou_values)

    best_threshold = list(THRESHOLDS)[best_index]

    best_iou = iou_values[best_index]

    # Save results
    all_iou_results.append(iou_values)

    best_thresholds.append(best_threshold)
    best_ious.append(best_iou)

    # ========================================================
    # PRINT RESULT
    # ========================================================

    print(
        f"Best threshold: {best_threshold}"
    )

    print(
        f"Best IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )

    # ========================================================
    # CREATE BEST PIPELINE
    # ========================================================

    # Threshold
    _, best_binary = cv2.threshold(
        gray,
        best_threshold,
        255,
        cv2.THRESH_BINARY_INV
    )

    # Closing
    best_closed = apply_closing(
        best_binary
    )

    # Median
    best_median = cv2.medianBlur(
        best_closed,
        MEDIAN_KERNEL_SIZE
    )

    # ========================================================
    # DISPLAY ALL STEPS
    # ========================================================

    plt.figure(figsize=(20, 5))

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 6, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"Original\n{image_path.name}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    plt.subplot(1, 6, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")

    # --------------------------------------------------------
    # THRESHOLD
    # --------------------------------------------------------

    plt.subplot(1, 6, 3)

    plt.imshow(
        best_binary,
        cmap="gray"
    )

    plt.title(
        f"Threshold\n"
        f"T = {best_threshold}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # CLOSING
    # --------------------------------------------------------

    plt.subplot(1, 6, 4)

    plt.imshow(
        best_closed,
        cmap="gray"
    )

    plt.title(
        f"Closing\n"
        f"Kernel = {CLOSING_KERNEL_SIZE}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # MEDIAN FILTER
    # --------------------------------------------------------

    plt.subplot(1, 6, 5)

    plt.imshow(
        best_median,
        cmap="gray"
    )

    plt.title(
        f"Median Filter\n"
        f"Kernel = {MEDIAN_KERNEL_SIZE}\n"
        f"IoU = {best_iou:.2%}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 6, 6)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # IoU VS THRESHOLD
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.plot(
        list(THRESHOLDS),
        iou_values
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.scatter(
        best_threshold,
        best_iou
    )

    plt.xlabel("Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs Threshold\n"
        f"{image_path.name}"
    )

    plt.grid()

    plt.show()


# ============================================================
# AVERAGE IoU
# ============================================================

if len(best_ious) > 0:

    average_iou = np.mean(
        best_ious
    )

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    print(
        f"Average IoU: {average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )

else:

    print("No IoU results available.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ============================================================
# SETTINGS
# ============================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)

NUMBER_OF_IMAGES = 10

# First threshold
THRESHOLDS = range(0, 256)

# Morphological Closing
CLOSING_KERNEL_SIZE = 3
CLOSING_ITERATIONS = 1


# ============================================================
# GET IMAGE FILES
# ============================================================

valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted([
    path
    for path in image_folder.glob("*")
    if path.suffix.lower() in valid_extensions
])

print("Total images found:", len(image_paths))


# ============================================================
# IoU FUNCTION
# ============================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# ============================================================
# MORPHOLOGICAL CLOSING
# ============================================================

def apply_closing(binary_image):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            CLOSING_KERNEL_SIZE,
            CLOSING_KERNEL_SIZE
        )
    )

    closed = cv2.morphologyEx(
        binary_image,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=CLOSING_ITERATIONS
    )

    return closed


# ============================================================
# STORE RESULTS
# ============================================================

best_thresholds = []
best_ious = []


# ============================================================
# PROCESS IMAGES
# ============================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    print("\n" + "=" * 70)
    print("IMAGE:", image_path.name)
    print("=" * 70)

    # --------------------------------------------------------
    # FIND MASK
    # --------------------------------------------------------

    mask_path = mask_folder / image_path.name

    if not mask_path.exists():
        print("Mask not found:", image_path.name)
        continue

    # --------------------------------------------------------
    # READ IMAGE
    # --------------------------------------------------------

    image = cv2.imread(str(image_path))

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print("Could not read:", image_path.name)
        continue

    # ========================================================
    # 1. GRAYSCALE
    # ========================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # ========================================================
    # GROUND TRUTH
    # ========================================================

    _, mask_binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    # ========================================================
    # TEST ALL FIRST THRESHOLDS
    # ========================================================

    iou_values = []

    for threshold in THRESHOLDS:

        # ====================================================
        # 2. FIRST THRESHOLD
        # ====================================================

        _, binary = cv2.threshold(
            gray,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )

        # ====================================================
        # 3. CLOSING
        # ====================================================

        closed = apply_closing(
            binary
        )

        # ====================================================
        # 4. OTSU
        # ====================================================

        _, otsu = cv2.threshold(
            closed,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # ====================================================
        # 5. IoU
        # ====================================================

        iou = calculate_iou(
            otsu,
            mask_binary
        )

        iou_values.append(iou)

    # ========================================================
    # FIND BEST FIRST THRESHOLD
    # ========================================================

    best_index = np.argmax(
        iou_values
    )

    best_threshold = list(
        THRESHOLDS
    )[best_index]

    best_iou = iou_values[
        best_index
    ]

    best_thresholds.append(
        best_threshold
    )

    best_ious.append(
        best_iou
    )

    # ========================================================
    # PRINT RESULT
    # ========================================================

    print(
        f"Best first threshold: "
        f"{best_threshold}"
    )

    print(
        f"Best IoU: "
        f"{best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )

    # ========================================================
    # CREATE BEST PIPELINE
    # ========================================================

    # First threshold
    _, best_binary = cv2.threshold(
        gray,
        best_threshold,
        255,
        cv2.THRESH_BINARY_INV
    )

    # Closing
    best_closed = apply_closing(
        best_binary
    )

    # Otsu
    otsu_threshold, best_otsu = cv2.threshold(
        best_closed,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    print(
        f"Otsu threshold: {otsu_threshold:.0f}"
    )

    # ========================================================
    # DISPLAY ALL STEPS
    # ========================================================

    plt.figure(figsize=(20, 5))

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    plt.subplot(1, 6, 1)

    plt.imshow(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"Original\n{image_path.name}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    plt.subplot(1, 6, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")

    # --------------------------------------------------------
    # FIRST THRESHOLD
    # --------------------------------------------------------

    plt.subplot(1, 6, 3)

    plt.imshow(
        best_binary,
        cmap="gray"
    )

    plt.title(
        f"Threshold\n"
        f"T = {best_threshold}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # CLOSING
    # --------------------------------------------------------

    plt.subplot(1, 6, 4)

    plt.imshow(
        best_closed,
        cmap="gray"
    )

    plt.title(
        f"Closing\n"
        f"Kernel = {CLOSING_KERNEL_SIZE}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # OTSU
    # --------------------------------------------------------

    plt.subplot(1, 6, 5)

    plt.imshow(
        best_otsu,
        cmap="gray"
    )

    plt.title(
        f"Otsu\n"
        f"T = {otsu_threshold:.0f}\n"
        f"IoU = {best_iou:.2%}"
    )

    plt.axis("off")

    # --------------------------------------------------------
    # GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 6, 6)

    plt.imshow(
        mask_binary,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # IoU VS FIRST THRESHOLD
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.plot(
        list(THRESHOLDS),
        iou_values
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.scatter(
        best_threshold,
        best_iou
    )

    plt.xlabel("First Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs First Threshold\n"
        f"{image_path.name}"
    )

    plt.grid()

    plt.show()


# ============================================================
# AVERAGE IoU
# ============================================================

if len(best_ious) > 0:

    average_iou = np.mean(
        best_ious
    )

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    print(
        f"Average IoU: "
        f"{average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )

else:

    print("No IoU results available.")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================

NUMBER_OF_IMAGES = 100

THRESHOLDS = range(0, 256)

BLACKHAT_KERNEL_SIZE = 15


# =========================================================
# PATHS
# =========================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


# =========================================================
# GET IMAGE PATHS
# =========================================================

image_paths = sorted(image_folder.glob("*.jpg"))


# =========================================================
# IoU FUNCTION
# =========================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# =========================================================
# BLACK-HAT KERNEL
# =========================================================

blackhat_kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (
        BLACKHAT_KERNEL_SIZE,
        BLACKHAT_KERNEL_SIZE
    )
)


# =========================================================
# STORE BEST IoUs
# =========================================================

best_ious = []


# =========================================================
# PROCESS IMAGES
# =========================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    # -----------------------------------------------------
    # Read image
    # -----------------------------------------------------

    image = cv2.imread(str(image_path))

    # Read ground-truth mask
    mask_path = mask_folder / image_path.name
    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print(f"Could not read: {image_path.name}")
        continue


    # -----------------------------------------------------
    # Convert to grayscale
    # -----------------------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )


    # -----------------------------------------------------
    # BLACK-HAT FILTER
    # -----------------------------------------------------
    # Black-hat = Closing - Original
    #
    # It highlights dark objects/details
    # on a brighter background.
    # -----------------------------------------------------

    blackhat = cv2.morphologyEx(
        gray,
        cv2.MORPH_BLACKHAT,
        blackhat_kernel
    )


    # -----------------------------------------------------
    # TEST ALL THRESHOLDS
    # -----------------------------------------------------

    ious = []

    best_iou = -1
    best_threshold = 0
    best_prediction = None

    for threshold in THRESHOLDS:

        # Black-hat produces bright pixels
        # where dark cracks are detected.
        _, prediction = cv2.threshold(
            blackhat,
            threshold,
            255,
            cv2.THRESH_BINARY
        )

        iou = calculate_iou(
            prediction,
            mask
        )

        ious.append(iou)

        # Keep best result
        if iou > best_iou:

            best_iou = iou
            best_threshold = threshold
            best_prediction = prediction.copy()


    # -----------------------------------------------------
    # SAVE BEST IoU
    # -----------------------------------------------------

    best_ious.append(best_iou)


    # -----------------------------------------------------
    # PRINT RESULT
    # -----------------------------------------------------

    print(
        f"{image_path.name} | "
        f"Best Threshold: {best_threshold} | "
        f"IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )


    # =====================================================
    # DISPLAY RESULTS
    # =====================================================

    plt.figure(figsize=(18, 5))


    # -----------------------------------------------------
    # Original
    # -----------------------------------------------------

    plt.subplot(1, 5, 1)

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.title("Original")

    plt.axis("off")


    # -----------------------------------------------------
    # Grayscale
    # -----------------------------------------------------

    plt.subplot(1, 5, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")


    # -----------------------------------------------------
    # Black-hat
    # -----------------------------------------------------

    plt.subplot(1, 5, 3)

    plt.imshow(
        blackhat,
        cmap="gray"
    )

    plt.title("Black-hat")

    plt.axis("off")


    # -----------------------------------------------------
    # Best Prediction
    # -----------------------------------------------------

    plt.subplot(1, 5, 4)

    plt.imshow(
        best_prediction,
        cmap="gray"
    )

    plt.title(
        f"Prediction\n"
        f"Threshold={best_threshold}\n"
        f"IoU={best_iou:.2%}"
    )

    plt.axis("off")


    # -----------------------------------------------------
    # Ground Truth
    # -----------------------------------------------------

    plt.subplot(1, 5, 5)

    plt.imshow(
        mask,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")


    plt.tight_layout()
    plt.show()


    # =====================================================
    # IoU VS THRESHOLD
    # =====================================================

    plt.figure(figsize=(8, 5))

    plt.plot(
        list(THRESHOLDS),
        ious
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.xlabel("Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs Threshold - {image_path.name}"
    )

    plt.grid()

    plt.show()


# =========================================================
# FINAL AVERAGE IoU
# =========================================================

if len(best_ious) > 0:

    average_iou = np.mean(best_ious)

    print("\n==============================")
    print("BLACK-HAT RESULTS")
    print("==============================")

    print(
        f"Average IoU: "
        f"{average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================

NUMBER_OF_IMAGES = 100

THRESHOLDS = range(0, 256)

GAUSSIAN_KERNEL_SIZE = 5

CLOSING_KERNEL_SIZE = 3
CLOSING_ITERATIONS = 1

MEDIAN_KERNEL_SIZE = 3


# =========================================================
# PATHS
# =========================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


# =========================================================
# GET IMAGE PATHS
# =========================================================

image_paths = sorted(image_folder.glob("*.jpg"))


# =========================================================
# IoU FUNCTION
# =========================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# =========================================================
# MORPHOLOGICAL CLOSING KERNEL
# =========================================================

closing_kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (
        CLOSING_KERNEL_SIZE,
        CLOSING_KERNEL_SIZE
    )
)


# =========================================================
# STORE BEST IoUs
# =========================================================

best_ious = []


# =========================================================
# PROCESS IMAGES
# =========================================================

for image_path in image_paths[:NUMBER_OF_IMAGES]:

    # -----------------------------------------------------
    # Read image
    # -----------------------------------------------------

    image = cv2.imread(str(image_path))

    mask_path = mask_folder / image_path.name

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None or mask is None:
        print(f"Could not read: {image_path.name}")
        continue


    # -----------------------------------------------------
    # Grayscale
    # -----------------------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )


    # -----------------------------------------------------
    # GAUSSIAN BLUR
    # -----------------------------------------------------

    blurred = cv2.GaussianBlur(
        gray,
        (
            GAUSSIAN_KERNEL_SIZE,
            GAUSSIAN_KERNEL_SIZE
        ),
        0
    )


    # -----------------------------------------------------
    # TEST ALL THRESHOLDS
    # -----------------------------------------------------

    ious = []

    best_iou = -1
    best_threshold = 0
    best_prediction = None

    for threshold in THRESHOLDS:

        # -------------------------------------------------
        # Threshold
        # Cracks are dark, so use BINARY_INV
        # -------------------------------------------------

        _, binary = cv2.threshold(
            blurred,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )


        # -------------------------------------------------
        # CLOSING
        # -------------------------------------------------

        closed = cv2.morphologyEx(
            binary,
            cv2.MORPH_CLOSE,
            closing_kernel,
            iterations=CLOSING_ITERATIONS
        )


        # -------------------------------------------------
        # MEDIAN FILTER
        # -------------------------------------------------

        prediction = cv2.medianBlur(
            closed,
            MEDIAN_KERNEL_SIZE
        )


        # -------------------------------------------------
        # IoU
        # -------------------------------------------------

        iou = calculate_iou(
            prediction,
            mask
        )

        ious.append(iou)


        # -------------------------------------------------
        # Save best result
        # -------------------------------------------------

        if iou > best_iou:

            best_iou = iou
            best_threshold = threshold
            best_prediction = prediction.copy()


    # =====================================================
    # SAVE BEST IoU
    # =====================================================

    best_ious.append(best_iou)


    # =====================================================
    # PRINT RESULT
    # =====================================================

    print(
        f"{image_path.name} | "
        f"Best Threshold: {best_threshold} | "
        f"IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )


    # =====================================================
    # DISPLAY PROCESSING STEPS
    # =====================================================

    plt.figure(figsize=(18, 5))


    # -----------------------------------------------------
    # Original
    # -----------------------------------------------------

    plt.subplot(1, 6, 1)

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.title("Original")

    plt.axis("off")


    # -----------------------------------------------------
    # Grayscale
    # -----------------------------------------------------

    plt.subplot(1, 6, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")


    # -----------------------------------------------------
    # Gaussian Blur
    # -----------------------------------------------------

    plt.subplot(1, 6, 3)

    plt.imshow(
        blurred,
        cmap="gray"
    )

    plt.title(
        f"Gaussian Blur\n"
        f"Kernel={GAUSSIAN_KERNEL_SIZE}"
    )

    plt.axis("off")


    # -----------------------------------------------------
    # Best Threshold Result
    # -----------------------------------------------------

    _, threshold_image = cv2.threshold(
        blurred,
        best_threshold,
        255,
        cv2.THRESH_BINARY_INV
    )

    plt.subplot(1, 6, 4)

    plt.imshow(
        threshold_image,
        cmap="gray"
    )

    plt.title(
        f"Threshold={best_threshold}"
    )

    plt.axis("off")


    # -----------------------------------------------------
    # Final Prediction
    # -----------------------------------------------------

    plt.subplot(1, 6, 5)

    plt.imshow(
        best_prediction,
        cmap="gray"
    )

    plt.title(
        f"Prediction\n"
        f"IoU={best_iou:.2%}"
    )

    plt.axis("off")


    # -----------------------------------------------------
    # Ground Truth
    # -----------------------------------------------------

    plt.subplot(1, 6, 6)

    plt.imshow(
        mask,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")


    plt.tight_layout()
    plt.show()


    # =====================================================
    # IoU VS THRESHOLD
    # =====================================================

    plt.figure(figsize=(8, 5))

    plt.plot(
        list(THRESHOLDS),
        ious
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.xlabel("Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs Threshold - {image_path.name}"
    )

    plt.grid()

    plt.show()


# =========================================================
# FINAL AVERAGE IoU
# =========================================================

if len(best_ious) > 0:

    average_iou = np.mean(best_ious)

    print("\n==============================")
    print("GAUSSIAN BLUR RESULTS")
    print("==============================")

    print(
        f"Average IoU: "
        f"{average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================

NUMBER_OF_IMAGES = 100

THRESHOLDS = range(0, 256)

# Bilateral Filter
BILATERAL_DIAMETER = 9
BILATERAL_SIGMA_COLOR = 75
BILATERAL_SIGMA_SPACE = 75

# Closing
CLOSING_KERNEL_SIZE = 3
CLOSING_ITERATIONS = 1

# Median Filter
MEDIAN_KERNEL_SIZE = 3


# =========================================================
# PATHS
# =========================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


# =========================================================
# GET IMAGE PATHS
# =========================================================

image_paths = sorted(image_folder.glob("*.jpg"))

print(f"Total images found: {len(image_paths)}")
print(f"Testing first {NUMBER_OF_IMAGES} images")


# =========================================================
# IoU FUNCTION
# =========================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# =========================================================
# CLOSING KERNEL
# =========================================================

closing_kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (
        CLOSING_KERNEL_SIZE,
        CLOSING_KERNEL_SIZE
    )
)


# =========================================================
# STORE BEST IoUs
# =========================================================

best_ious = []


# =========================================================
# PROCESS FIRST 100 IMAGES
# =========================================================

for image_number, image_path in enumerate(
    image_paths[:NUMBER_OF_IMAGES],
    start=1
):

    # -----------------------------------------------------
    # READ IMAGE
    # -----------------------------------------------------

    image = cv2.imread(str(image_path))

    # Matching mask
    mask_path = mask_folder / image_path.name

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        print(f"Could not read image: {image_path.name}")
        continue

    if mask is None:
        print(f"Could not read mask: {mask_path}")
        continue


    # -----------------------------------------------------
    # GRAYSCALE
    # -----------------------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )


    # -----------------------------------------------------
    # BILATERAL FILTER
    # -----------------------------------------------------

    filtered = cv2.bilateralFilter(
        gray,
        BILATERAL_DIAMETER,
        BILATERAL_SIGMA_COLOR,
        BILATERAL_SIGMA_SPACE
    )


    # -----------------------------------------------------
    # TEST ALL THRESHOLDS
    # -----------------------------------------------------

    ious = []

    best_iou = -1
    best_threshold = 0
    best_prediction = None

    for threshold in THRESHOLDS:

        # -------------------------------------------------
        # THRESHOLD
        #
        # Cracks are dark, therefore BINARY_INV
        # -------------------------------------------------

        _, binary = cv2.threshold(
            filtered,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )


        # -------------------------------------------------
        # MORPHOLOGICAL CLOSING
        # -------------------------------------------------

        closed = cv2.morphologyEx(
            binary,
            cv2.MORPH_CLOSE,
            closing_kernel,
            iterations=CLOSING_ITERATIONS
        )


        # -------------------------------------------------
        # MEDIAN FILTER
        # -------------------------------------------------

        prediction = cv2.medianBlur(
            closed,
            MEDIAN_KERNEL_SIZE
        )


        # -------------------------------------------------
        # CALCULATE IoU
        # -------------------------------------------------

        iou = calculate_iou(
            prediction,
            mask
        )

        ious.append(iou)


        # -------------------------------------------------
        # SAVE BEST RESULT
        # -------------------------------------------------

        if iou > best_iou:

            best_iou = iou
            best_threshold = threshold
            best_prediction = prediction.copy()


    # =====================================================
    # SAVE BEST IoU
    # =====================================================

    best_ious.append(best_iou)


    # =====================================================
    # PRINT RESULT
    # =====================================================

    print(
        f"[{image_number:03d}/{NUMBER_OF_IMAGES}] "
        f"{image_path.name} | "
        f"Best Threshold: {best_threshold} | "
        f"IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )


    # =====================================================
    # CREATE BEST THRESHOLD IMAGE
    # =====================================================

    _, threshold_image = cv2.threshold(
        filtered,
        best_threshold,
        255,
        cv2.THRESH_BINARY_INV
    )


    # =====================================================
    # CREATE BEST CLOSING IMAGE
    # =====================================================

    best_closed = cv2.morphologyEx(
        threshold_image,
        cv2.MORPH_CLOSE,
        closing_kernel,
        iterations=CLOSING_ITERATIONS
    )


    # =====================================================
    # DISPLAY PROCESSING STEPS
    # =====================================================

    plt.figure(figsize=(20, 5))


    # -----------------------------------------------------
    # 1. Original
    # -----------------------------------------------------

    plt.subplot(1, 7, 1)

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.title("Original")

    plt.axis("off")


    # -----------------------------------------------------
    # 2. Grayscale
    # -----------------------------------------------------

    plt.subplot(1, 7, 2)

    plt.imshow(
        gray,
        cmap="gray"
    )

    plt.title("Grayscale")

    plt.axis("off")


    # -----------------------------------------------------
    # 3. Bilateral Filter
    # -----------------------------------------------------

    plt.subplot(1, 7, 3)

    plt.imshow(
        filtered,
        cmap="gray"
    )

    plt.title("Bilateral Filter")

    plt.axis("off")


    # -----------------------------------------------------
    # 4. Threshold
    # -----------------------------------------------------

    plt.subplot(1, 7, 4)

    plt.imshow(
        threshold_image,
        cmap="gray"
    )

    plt.title(
        f"Threshold\n{best_threshold}"
    )

    plt.axis("off")


    # -----------------------------------------------------
    # 5. Closing
    # -----------------------------------------------------

    plt.subplot(1, 7, 5)

    plt.imshow(
        best_closed,
        cmap="gray"
    )

    plt.title("Closing")

    plt.axis("off")


    # -----------------------------------------------------
    # 6. Final Prediction
    # -----------------------------------------------------

    plt.subplot(1, 7, 6)

    plt.imshow(
        best_prediction,
        cmap="gray"
    )

    plt.title(
        f"Prediction\n"
        f"IoU = {best_iou:.2%}"
    )

    plt.axis("off")


    # -----------------------------------------------------
    # 7. Ground Truth
    # -----------------------------------------------------

    plt.subplot(1, 7, 7)

    plt.imshow(
        mask,
        cmap="gray"
    )

    plt.title("Ground Truth")

    plt.axis("off")


    plt.tight_layout()
    plt.show()


    # =====================================================
    # IoU VS THRESHOLD
    # =====================================================

    plt.figure(figsize=(8, 5))

    plt.plot(
        list(THRESHOLDS),
        ious
    )

    plt.axvline(
        best_threshold,
        linestyle="--"
    )

    plt.xlabel("Threshold")
    plt.ylabel("IoU")

    plt.title(
        f"IoU vs Threshold\n{image_path.name}"
    )

    plt.grid()

    plt.show()


# =========================================================
# FINAL AVERAGE IoU
# =========================================================

if len(best_ious) > 0:

    average_iou = np.mean(best_ious)

    print("\n")
    print("==============================")
    print("BILATERAL FILTER RESULTS")
    print("==============================")

    print(
        f"Images tested: {len(best_ious)}"
    )

    print(
        f"Average IoU: "
        f"{average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================

NUMBER_OF_IMAGES = 100

# Number of images to visually display
SHOW_IMAGES = 15

# Test every threshold
THRESHOLDS = range(0, 256)

# Median filter
MEDIAN_KERNEL_SIZE = 3


# =========================================================
# PATHS
# =========================================================

image_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\images\images"
)

mask_folder = Path(
    r"\\192.168.1.10\shear all\day3\train\masks\masks"
)


# =========================================================
# GET IMAGE PATHS
# =========================================================

image_paths = sorted(
    image_folder.glob("*.jpg")
)

print(f"Total images found: {len(image_paths)}")
print(f"Testing first {NUMBER_OF_IMAGES} images")


# =========================================================
# IoU FUNCTION
# =========================================================

def calculate_iou(prediction, mask):

    TP = np.logical_and(
        mask == 255,
        prediction == 255
    ).sum()

    FP = np.logical_and(
        mask == 0,
        prediction == 255
    ).sum()

    FN = np.logical_and(
        mask == 255,
        prediction == 0
    ).sum()

    denominator = TP + FP + FN

    if denominator == 0:
        return 0.0

    return TP / denominator


# =========================================================
# STORE BEST IoUs
# =========================================================

best_ious = []


# =========================================================
# PROCESS FIRST 100 IMAGES
# =========================================================

for image_number, image_path in enumerate(
    image_paths[:NUMBER_OF_IMAGES],
    start=1
):

    # =====================================================
    # READ IMAGE
    # =====================================================

    image = cv2.imread(
        str(image_path)
    )

    # Matching mask
    mask_path = mask_folder / image_path.name

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )


    # -----------------------------------------------------
    # Check image
    # -----------------------------------------------------

    if image is None:

        print(
            f"Could not read image: "
            f"{image_path.name}"
        )

        continue


    # -----------------------------------------------------
    # Check mask
    # -----------------------------------------------------

    if mask is None:

        print(
            f"Could not read mask: "
            f"{mask_path}"
        )

        continue


    # =====================================================
    # GRAYSCALE
    # =====================================================

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )


    # =====================================================
    # MEDIAN FILTER
    # =====================================================

    median = cv2.medianBlur(
        gray,
        MEDIAN_KERNEL_SIZE
    )


    # =====================================================
    # TEST ALL THRESHOLDS
    # =====================================================

    ious = []

    best_iou = -1
    best_threshold = 0
    best_prediction = None


    for threshold in THRESHOLDS:

        # -------------------------------------------------
        # Threshold
        #
        # Cracks are dark
        # Therefore use BINARY_INV
        # -------------------------------------------------

        _, prediction = cv2.threshold(
            median,
            threshold,
            255,
            cv2.THRESH_BINARY_INV
        )


        # -------------------------------------------------
        # Calculate IoU
        # -------------------------------------------------

        iou = calculate_iou(
            prediction,
            mask
        )

        ious.append(iou)


        # -------------------------------------------------
        # Save best result
        # -------------------------------------------------

        if iou > best_iou:

            best_iou = iou

            best_threshold = threshold

            best_prediction = prediction.copy()


    # =====================================================
    # SAVE BEST IoU
    # =====================================================

    best_ious.append(
        best_iou
    )


    # =====================================================
    # PRINT RESULT
    # =====================================================

    print(
        f"[{image_number:03d}/{NUMBER_OF_IMAGES}] "
        f"{image_path.name} | "
        f"Best Threshold: {best_threshold} | "
        f"IoU: {best_iou:.4f} "
        f"({best_iou * 100:.2f}%)"
    )


    # =====================================================
    # VISUALIZATION
    # =====================================================

    if image_number <= SHOW_IMAGES:

        # -------------------------------------------------
        # Create best threshold image
        # -------------------------------------------------

        _, threshold_image = cv2.threshold(
            median,
            best_threshold,
            255,
            cv2.THRESH_BINARY_INV
        )


        # -------------------------------------------------
        # Show processing stages
        # -------------------------------------------------

        plt.figure(
            figsize=(18, 5)
        )


        # =================================================
        # 1. ORIGINAL
        # =================================================

        plt.subplot(1, 6, 1)

        plt.imshow(
            cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )
        )

        plt.title("Original")

        plt.axis("off")


        # =================================================
        # 2. GRAYSCALE
        # =================================================

        plt.subplot(1, 6, 2)

        plt.imshow(
            gray,
            cmap="gray"
        )

        plt.title("Grayscale")

        plt.axis("off")


        # =================================================
        # 3. MEDIAN FILTER
        # =================================================

        plt.subplot(1, 6, 3)

        plt.imshow(
            median,
            cmap="gray"
        )

        plt.title(
            f"Median Filter\n"
            f"Kernel = {MEDIAN_KERNEL_SIZE}"
        )

        plt.axis("off")


        # =================================================
        # 4. THRESHOLD
        # =================================================

        plt.subplot(1, 6, 4)

        plt.imshow(
            threshold_image,
            cmap="gray"
        )

        plt.title(
            f"Threshold\n"
            f"{best_threshold}"
        )

        plt.axis("off")


        # =================================================
        # 5. PREDICTION
        # =================================================

        plt.subplot(1, 6, 5)

        plt.imshow(
            best_prediction,
            cmap="gray"
        )

        plt.title(
            f"Prediction\n"
            f"IoU = {best_iou:.2%}"
        )

        plt.axis("off")


        # =================================================
        # 6. GROUND TRUTH
        # =================================================

        plt.subplot(1, 6, 6)

        plt.imshow(
            mask,
            cmap="gray"
        )

        plt.title(
            "Ground Truth"
        )

        plt.axis("off")


        plt.tight_layout()

        plt.show()


        # =================================================
        # IoU VS THRESHOLD
        # =================================================

        plt.figure(
            figsize=(8, 5)
        )

        plt.plot(
            list(THRESHOLDS),
            ious
        )

        plt.axvline(
            best_threshold,
            linestyle="--"
        )

        plt.xlabel(
            "Threshold"
        )

        plt.ylabel(
            "IoU"
        )

        plt.title(
            f"IoU vs Threshold\n"
            f"{image_path.name}"
        )

        plt.grid()

        plt.show()


# =========================================================
# FINAL AVERAGE IoU
# =========================================================

if len(best_ious) > 0:

    average_iou = np.mean(
        best_ious
    )

    print("\n")
    print("==============================")
    print("MEDIAN FILTER RESULTS")
    print("==============================")

    print(
        f"Images tested: "
        f"{len(best_ious)}"
    )

    print(
        f"Average IoU: "
        f"{average_iou:.4f} "
        f"({average_iou * 100:.2f}%)"
    )